<img src="https://github.com/Multiomics-Analytics-Group/networks_to_study_microbes/blob/main/figures/cfb.png?raw=1" width="300">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Multiomics-Analytics-Group/networks_to_study_microbes/blob/main/notebooks/03_Databases_&_Data_Annotation/06_uniprot_api.ipynb)

# Networks to Study Microbes



# Data Annotation -- Use Case 1

In this notebook, we will use [UniProt's API](https://www.uniprot.org/help/programmatic_access) to get gene/protein information to annotate our experimental data.


As a project dataset, we will use [Xia et al 2022](https://www.nature.com/articles/s41467-022-30513-2): **Proteome allocations change linearly with the specific growth rate of Saccharomyces cerevisiae under glucose limitation**

<div>
<img src="https://github.com/biosustain/data_club/raw/main/figures/xia_et_al_2022.png" width="900"/>
</div>



And specifically the absolute proteome and transcriptome:

<div>
<img src="https://github.com/biosustain/data_club/raw/main/figures/xia_datasets.png" width="500"/>
</div>


In [10]:
import os
import requests, sys
import json
import pandas as pd

In [11]:
transcriptome_df = pd.read_csv("https://raw.githubusercontent.com/Multiomics-Analytics-Group/networks_to_study_microbes/refs/heads/UdeA2024/example_data/Xia_et_al_2022/transcriptomics.tsv", sep='\t', index_col=False)

In [12]:
transcriptome_df.head()

,mRNA,0.027 h-1,0.044 h-1,0.102 h-1,0.152 h-1,0.214 h-1,0.254 h-1,0.284 h-1,0.334 h-1,0.379 h-1
0,R0010W,74.443125,76.433295,48.458353,49.549858,54.371609,54.687635,64.371427,32.918948,41.388778
1,YLR155C,37.844620,36.332161,35.091533,31.096557,29.066724,26.818775,29.592791,23.029033,22.431883
2,YLR159W,7.223876,6.016159,7.014012,5.542861,4.800111,3.934089,4.485135,3.743151,2.596469
3,YHR056C,18.646820,14.038290,14.158659,14.748175,15.365871,13.534869,13.671023,14.309881,13.978857
4,R0030W,209.821451,223.130456,128.572630,122.797999,134.245757,119.268918,116.934733,37.133109,49.136868


In [19]:
transcriptome_df = transcriptome_df.head(n=20)

In [20]:
def get_accession(gene_name):
    requestURL = f"https://www.ebi.ac.uk/proteins/api/proteins?offset=0&size=100&gene={gene_name}"
    
    r = requests.get(requestURL, headers={ "Accept" : "application/json"})

    if not r.ok:
      r.raise_for_status()
      sys.exit()

    responseBody = r.text
    try:
        json_response = json.loads(responseBody)[0]["accession"]
    except:
        json_response = gene_name

    return json_response

In [21]:
#get_accession(gene_name="R0010W")

In [22]:
def get_protein_info(accession):
    requestURL = f"https://www.ebi.ac.uk/proteins/api/proteins/{accession}"

    r = requests.get(requestURL, headers={ "Accept" : "application/json"})

    if not r.ok:
      r.raise_for_status()
      sys.exit()
    
    responseBody = r.text
    try:
        json_response = json.loads(responseBody)
    except:
        json_response = {}

    return json_response

def extract_protein_info(accession, response):
    df =pd.DataFrame({"id":response["id"], "taxid":response["organism"]["taxonomy"], 
                      "organism":str(response["organism"]["names"][0]["value"]), 
                      "comments": response["comments"][0]["text"][0]["value"], 
                      "sequence":response["sequence"]["sequence"], 
                      "sequence_length":response["sequence"]["length"], 
                      "sequence_mass":response["sequence"]["mass"]}, index=[accession])
    return(df)

In [23]:
#result = get_protein_info(accession="P03870")

#extract_protein_info(accession="P03870", response=result)


In [24]:
transcriptome_df["Accessions"] = transcriptome_df["mRNA"].apply(lambda x: get_accession(gene_name=x))

In [25]:
transcriptome_df.to_csv("transcriptomics_mapped.tsv", sep='\t', index=False, doublequote=None, header=True)